In [1]:
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Timestamp:", datetime.now(timezone.utc).isoformat())

commands = [
    ["rocminfo"],
    ["rocm-smi"],
]

for command in commands:
    print(f"\n$ {' '.join(command)}")
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=60,
        check=False,
    )
    print(result.stdout[:10000])
    if result.stderr:
        print("STDERR:", result.stderr[:2000])
        

Python: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
Platform: Linux-6.8.0-79-generic-x86_64-with-glibc2.35
Timestamp: 2026-07-11T19:19:00.683334+00:00

$ rocminfo
ROCk module version 6.16.13 is loaded
HSA System Attributes    
Runtime Version:         1.18
Runtime Ext Version:     1.15
System Timestamp Freq.:  1000.000000MHz
Sig. Max Wait Duration:  18446744073709551615 (0xFFFFFFFFFFFFFFFF) (timestamp count)
Machine Model:           LARGE                              
System Endianness:       LITTLE                             
Mwaitx:                  DISABLED
XNACK enabled:           NO
DMAbuf Support:          YES
VMM Support:             YES

HSA Agents               
*******                  
Agent 1                  
*******                  
  Name:                    AMD EPYC 9334 32-Core Processor    
  Uuid:                    CPU-XX                             
  Marketing Name:          AMD EPYC 9334 32-Core Processor    
  Vendor Name:             CPU               

In [2]:
!rocm-smi



======================================== ROCm System Management Interface ========================================
================================================== Concise Info ==================================================
Device  Node  IDs              Temp    Power  Partitions          SCLK  MCLK   Fan    Perf  PwrCap  VRAM%  GPU%  
              (DID,     GUID)  (Edge)  (Avg)  (Mem, Compute, ID)                                                 
0       4     0x744b,   60148  27.0°C  9.0W   N/A, N/A, 0         0Mhz  96Mhz  20.0%  auto  241.0W  0%     0%    
============================================== End of ROCm SMI Log ===============================================


In [3]:
import torch

print("PyTorch version:", torch.__version__)
print("ROCm/HIP version:", torch.version.hip)
print("GPU available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))


PyTorch version: 2.9.1+gitff65f5b
ROCm/HIP version: 7.2.53211-e1a6bc5663
GPU available: True
GPU count: 1
GPU name: 
GPU capability: (11, 0)


In [4]:
%pip install sentence-transformers

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 10.0 MB/s  0:00:00

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /opt/venv/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from sentence_transformers import SentenceTransformer
import torch
import time

device = "cuda"

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device,
)

topics = [
    "Cell Structure",
    "Photosynthesis",
    "Aerobic Respiration",
    "DNA Replication",
    "Protein Synthesis",
]

start = time.time()

embeddings = model.encode(
    topics,
    normalize_embeddings=True,
)

print("Embedding shape:", embeddings.shape)
print("Time:", time.time()-start)
print("GPU:", torch.cuda.is_available())
print("HIP:", torch.version.hip)

/opt/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[aiter] WARNING: NUMA balancing is enabled, which may cause errors. It is recommended to disable NUMA balancing by running "sudo sh -c 'echo 0 > /proc/sys/kernel/numa_balancing'" for more details: https://rocm.docs.amd.com/en/latest/how-to/system-optimization/mi300x.html#disable-numa-auto-balancing
[aiter] start build [module_aiter_enum] under /opt/venv/lib/python3.10/site-packages/aiter/jit/build/module_aiter_enum
clang (LLVM option parsing): Unknown command line argument '-amdgpu-coerce-illegal-types=1'.  Try: 'clang (LLVM option parsing) --help'
clang (LLVM option parsing): Did you mean '--amdgpu-mode-register=1'?
failed to execute:/opt/rocm/lib/llvm/bin/clang++  --offload-arch=gfx1100 -O3  -mllvm -amdgpu-coerce-illegal-types=1 -x hip -E -P /d

Embedding shape: (5, 384)
Time: 1.3746838569641113
GPU: True
HIP: 7.2.53211-e1a6bc5663


In [6]:
from sentence_transformers import SentenceTransformer
import torch

print("Loading model...")

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

print("Model loaded!")

embeddings = model.encode(
    [
        "Cell Structure",
        "Photosynthesis",
        "DNA Replication"
    ],
    show_progress_bar=False,
)

print("Embedding shape:", embeddings.shape)
print("GPU available:", torch.cuda.is_available())
print("HIP:", torch.version.hip)

Loading model...
Model loaded!
Embedding shape: (3, 384)
GPU available: True
HIP: 7.2.53211-e1a6bc5663


In [7]:
import json
import torch
from datetime import datetime, timezone

manifest = {
    "project": "Rebound",
    "operation": "Semantic topic embedding generation",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "torch_version": torch.__version__,
    "rocm_hip_version": torch.version.hip,
    "gpu_available": torch.cuda.is_available(),
    "embedding_dimensions": 384,
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "status": "completed"
}

with open("amd_run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("AMD manifest saved.")


AMD manifest saved.


In [8]:
!git clone https://github.com/cyeeezz/rebound-amd.git
%cd rebound-amd

Cloning into 'rebound-amd'...
fatal: unable to access 'https://github.com/cyeeezz/rebound-amd.git/': server certificate verification failed. CAfile: none CRLfile: none
[Errno 2] No such file or directory: 'rebound-amd'
/workspace


/opt/venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [9]:
print("Hello")

Hello


In [10]:
!wget --no-check-certificate https://raw.githubusercontent.com/cyeeezz/rebound-amd/main/knowledge_map.json

--2026-07-11 19:41:34--  https://raw.githubusercontent.com/cyeeezz/rebound-amd/main/knowledge_map.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... failed: Connection timed out.
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... failed: Connection timed out.
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... failed: Connection timed out.
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 11624 (11K) [text/plain]
Saving to: ‘knowledge_map.json.1’

knowledge_map.json. 100%[===================>]  11.35K  --.-KB/s    in 0.004s  

2026-07-11 19:48:20 (2.64 MB/s) - ‘knowledge_map.json.1’ saved [11624/11624]



In [11]:
!ping -c 3 github.com

/bin/bash: line 1: ping: command not found


In [12]:
!curl -I https://raw.githubusercontent.com

HTTP/2 301 
content-security-policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
location: https://github.com/
strict-transport-security: max-age=31536000
x-content-type-options: nosniff
x-frame-options: deny
x-xss-protection: 1; mode=block
x-github-request-id: DD30:1EB8:239AF8:4897C5:6A5298C1
accept-ranges: bytes
date: Sat, 11 Jul 2026 19:48:21 GMT
via: 1.1 varnish
x-served-by: cache-sin-wsss1830095-SIN
x-cache: HIT
x-cache-hits: 10
x-timer: S1783799301.367545,VS0,VE0
vary: Authorization,Accept-Encoding
access-control-allow-origin: *
cross-origin-resource-policy: cross-origin
x-fastly-request-id: 3623fe6b7ea6850d8724c602a2d7dbf2a566bac0
expires: Sat, 11 Jul 2026 19:53:21 GMT
source-age: 1345
content-length: 0



In [13]:
topics = [
    {
        "title": "Cell Structure",
        "summary": "Structure and function of organelles."
    },
    {
        "title": "Photosynthesis",
        "summary": "Light-dependent and Calvin cycle reactions."
    },
    {
        "title": "Cellular Respiration",
        "summary": "ATP production through glycolysis, Krebs cycle and ETC."
    },
    {
        "title": "DNA Replication",
        "summary": "Semi-conservative DNA replication."
    },
    {
        "title": "Protein Synthesis",
        "summary": "Transcription and translation."
    },
    {
        "title": "Genetics",
        "summary": "Inheritance and Mendelian genetics."
    },
    {
        "title": "Evolution",
        "summary": "Natural selection and speciation."
    }
]


In [14]:
import json

with open("knowledge_map.json", "r", encoding="utf-8") as f:
    knowledge = json.load(f)

topics = knowledge["topics"]

print(f"Loaded {len(topics)} Rebound topics")

Loaded 7 Rebound topics


In [15]:
def build_topic_text(topic):
    title = topic.get("title", "")
    summary = topic.get("summary", "")

    concepts = topic.get("key_concepts", [])
    if isinstance(concepts, list):
        concepts = ", ".join(concepts)
    else:
        concepts = ""

    return (
        f"Topic: {title}. "
        f"Summary: {summary}. "
        f"Key concepts: {concepts}."
    )

topic_texts = [build_topic_text(topic) for topic in topics]

print("Example input to embedding model:\n")
print(topic_texts[0])

Example input to embedding model:

Topic: Cell Structure. Summary: Learning objectives
● Identify the main organelles found in eukaryotic cells and state their functions. ● Compare prokaryotic and eukaryotic cells.. Key concepts: Cells, Mitochondria, Ribosomes, Learning, Identify, Compare.


In [16]:
import torch
import time
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading model...")

model = SentenceTransformer(
    MODEL_ID,
    device="cuda"
)

print("Model loaded!")

torch.cuda.empty_cache()
torch.cuda.synchronize()

start = time.perf_counter()

embeddings = model.encode(
    topic_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

torch.cuda.synchronize()

elapsed = time.perf_counter() - start

print("Embedding shape:", embeddings.shape)
print("Time:", elapsed)
print("GPU:", torch.cuda.is_available())
print("HIP:", torch.version.hip)

Loading model...
Model loaded!
Embedding shape: (7, 384)
Time: 0.010223845019936562
GPU: True
HIP: 7.2.53211-e1a6bc5663


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(embeddings)

SEMANTIC_THRESHOLD = 0.30
MAX_RELATIONSHIPS_PER_TOPIC = 3

semantic_relationships = []

for source_index, source_topic in enumerate(topics):
    candidates = []

    for target_index, target_topic in enumerate(topics):
        if source_index == target_index:
            continue

        score = float(similarity_matrix[source_index, target_index])

        if score >= SEMANTIC_THRESHOLD:
            candidates.append(
                {
                    "target": target_topic.get("title", "Untitled"),
                    "similarity": round(score, 4),
                }
            )

    candidates.sort(
        key=lambda item: item["similarity"],
        reverse=True,
    )

    for candidate in candidates[:MAX_RELATIONSHIPS_PER_TOPIC]:
        semantic_relationships.append(
            {
                "source": source_topic.get("title", "Untitled"),
                "target": candidate["target"],
                "similarity": candidate["similarity"],
                "relationship_type": "semantic_similarity",
            }
        )

print("Semantic relationships generated:", len(semantic_relationships))

for relationship in semantic_relationships:
    print(
        f"{relationship['source']} → "
        f"{relationship['target']} "
        f"({relationship['similarity']:.1%})"
    )

Semantic relationships generated: 21
Cell Structure → Gene Expression (50.9%)
Cell Structure → DNA Structure (50.1%)
Cell Structure → Cellular Respiration (48.5%)
Cellular Respiration → Energy Transfer in Cells (63.9%)
Cellular Respiration → Gene Expression (50.7%)
Cellular Respiration → Cell Structure (48.5%)
Energy Transfer in Cells → Cellular Respiration (63.9%)
Energy Transfer in Cells → Gene Expression (48.5%)
Energy Transfer in Cells → Cell Structure (47.5%)
DNA Structure → Gene Expression (51.0%)
DNA Structure → Cell Structure (50.1%)
DNA Structure → Genetic Inheritance (44.3%)
Gene Expression → DNA Structure (51.0%)
Gene Expression → Cell Structure (50.9%)
Gene Expression → Cellular Respiration (50.7%)
Genetic Inheritance → DNA Structure (44.3%)
Genetic Inheritance → Historical Biology Experiments (44.0%)
Genetic Inheritance → Gene Expression (43.5%)
Historical Biology Experiments → Genetic Inheritance (44.0%)
Historical Biology Experiments → DNA Structure (40.4%)
Historical Bi

In [18]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

artifact_dir = Path("amd_artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

embeddings_path = artifact_dir / "embeddings.npy"
metadata_path = artifact_dir / "topic_metadata.json"
manifest_path = artifact_dir / "amd_run_manifest.json"

np.save(
    embeddings_path,
    embeddings,
    allow_pickle=False,
)

topic_metadata = []

for index, topic in enumerate(topics):
    title = str(topic.get("title", "Untitled"))

    related_topics = [
        relationship
        for relationship in semantic_relationships
        if relationship["source"] == title
    ]

    topic_metadata.append(
        {
            "index": index,
            "title": title,
            "summary": str(topic.get("summary", "")),
            "key_concepts": topic.get("key_concepts", []),
            "semantic_relationships": related_topics,
        }
    )

metadata_payload = {
    "project": "Rebound",
    "input_source": "knowledge_map.json",
    "model_id": MODEL_ID,
    "topic_count": len(topic_metadata),
    "embedding_dimensions": int(embeddings.shape[1]),
    "topics": topic_metadata,
    "relationships": semantic_relationships,
}

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(
        metadata_payload,
        file,
        indent=2,
        ensure_ascii=False,
    )

device_name = torch.cuda.get_device_name(0) or "AMD ROCm device 0"

manifest = {
    "project": "Rebound",
    "operation": (
        "Semantic topic embedding generation and "
        "relationship discovery"
    ),
    "execution_timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "execution_status": "completed",
    "input_source": "knowledge_map.json",
    "platform": platform.platform(),
    "framework": "PyTorch and Sentence Transformers",
    "torch_version": torch.__version__,
    "rocm_hip_version": torch.version.hip,
    "gpu_available": bool(torch.cuda.is_available()),
    "gpu_count": int(torch.cuda.device_count()),
    "device_name": device_name,
    "model_id": MODEL_ID,
    "topic_count": len(topics),
    "embedding_dimensions": int(embeddings.shape[1]),
    "relationship_count": len(semantic_relationships),
    "semantic_threshold": SEMANTIC_THRESHOLD,
    "execution_seconds": round(elapsed, 6),
    "notebook_path": "notebooks/rebound_amd_embeddings.ipynb",
    "artifact_files": [
        "amd_artifacts/embeddings.npy",
        "amd_artifacts/topic_metadata.json",
        "amd_artifacts/amd_run_manifest.json",
    ],
}

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(json.dumps(manifest, indent=2))

{
  "project": "Rebound",
  "operation": "Semantic topic embedding generation and relationship discovery",
  "execution_timestamp_utc": "2026-07-11T19:50:45.441199+00:00",
  "execution_status": "completed",
  "input_source": "knowledge_map.json",
  "platform": "Linux-6.8.0-79-generic-x86_64-with-glibc2.35",
  "framework": "PyTorch and Sentence Transformers",
  "torch_version": "2.9.1+gitff65f5b",
  "rocm_hip_version": "7.2.53211-e1a6bc5663",
  "gpu_available": true,
  "gpu_count": 1,
  "device_name": "AMD ROCm device 0",
  "model_id": "sentence-transformers/all-MiniLM-L6-v2",
  "topic_count": 7,
  "embedding_dimensions": 384,
  "relationship_count": 21,
  "semantic_threshold": 0.3,
  "execution_seconds": 0.010224,
  "notebook_path": "notebooks/rebound_amd_embeddings.ipynb",
  "artifact_files": [
    "amd_artifacts/embeddings.npy",
    "amd_artifacts/topic_metadata.json",
    "amd_artifacts/amd_run_manifest.json"
  ]
}


In [19]:
loaded_embeddings = np.load(
    embeddings_path,
    allow_pickle=False,
)

with metadata_path.open("r", encoding="utf-8") as file:
    loaded_metadata = json.load(file)

with manifest_path.open("r", encoding="utf-8") as file:
    loaded_manifest = json.load(file)

print("Embeddings shape:", loaded_embeddings.shape)
print("Topics:", len(loaded_metadata["topics"]))
print("Relationships:", len(loaded_metadata["relationships"]))
print("Manifest status:", loaded_manifest["execution_status"])
print("Device:", loaded_manifest["device_name"])

for path in [
    embeddings_path,
    metadata_path,
    manifest_path,
]:
    print(
        path,
        "| exists:",
        path.exists(),
        "| bytes:",
        path.stat().st_size,
    )

Embeddings shape: (7, 384)
Topics: 7
Relationships: 21
Manifest status: completed
Device: AMD ROCm device 0
amd_artifacts/embeddings.npy | exists: True | bytes: 10880
amd_artifacts/topic_metadata.json | exists: True | bytes: 10781
amd_artifacts/amd_run_manifest.json | exists: True | bytes: 931
